In [2]:
#Définir un état personnalisé d’agent en héritant de AgentState
from langchain.agents import AgentState
class CustomState(AgentState):
    favourite_colour: str

In [3]:
#Agent qui modifie un état
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langchain_ollama import ChatOllama
from langgraph.types import Command
from langchain.messages import HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

model = ChatOllama(
model="llama3.2:3b", # ou mistral, gemma, etc.
)

@tool
def update_favourite_colour(
    favourite_colour: str,
    runtime: ToolRuntime,
) -> Command:
    """
    Update the user's favourite colour in the agent state.

    Args:
        favourite_colour (str): The user's favourite colour.
        runtime (ToolRuntime): Runtime information for the current tool call.

    Returns:
        Command: A command that updates the agent state.
    """
    return Command(
        update={
            "favourite_colour": favourite_colour,
            "messages": [
                ToolMessage(
                    content="Successfully updated favourite colour.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# Create the agent
agent = create_agent(
    model=model,
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState,
)


# Invoke the agent
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="My favourite colour is green."
            )
        ]
    },
    config={
        "configurable": {
            "thread_id": "1",
        }
    },
)


# Display the final response
print(response["messages"][-1].content)

Your favourite colour has been successfully updated to green! Is there anything else I can help you with?


In [4]:
# Agent qui récupère un état

@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"


agent = create_agent(
    model=model,
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="My favourite colour is green")
        ]
    },
    {
        "configurable": {
            "thread_id": "1"
        }
    }
)

print(response["messages"][-1].content)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What's my favourite colour?")
        ]
    },
    {
        "configurable": {
            "thread_id": "1"
        }
    }
)

print(response["messages"][-1].content)

It seems that I've successfully stored your favourite colour in our database. If you'd like to ask another question or request a service, feel free to do so!
Your favourite colour is indeed green. Is there anything else I can help you with?
